In [20]:
import pandas as pd

This notebook will add coauthorship links given a folder containing a network that was outlined earlier.

In [31]:
path_to_folder = "../CFDE/"  

In [32]:
for_edge = pd.read_csv(path_to_folder + "authors.published.pmids.edges.csv")

# Let us create a dictionary that maps each author to their respective pmids.
source_to_targets = for_edge.groupby('source')['target'].apply(set).to_dict()


#Create an inverted dictionary mapping targets to sources
target_to_sources = {}
for source, targets in source_to_targets.items():
    for target in targets:
        target_to_sources.setdefault(target, set()).add(source)

# Create the final dictionary mapping each source to all other sources that share at least one target
source_to_related_sources = {}
for source, targets in source_to_targets.items():
    related_sources = set()
    for target in targets:
        related_sources.update(target_to_sources[target])
    related_sources.discard(source)  # Remove the source itself
    source_to_related_sources[source] = related_sources

print(source_to_related_sources)


{0: {1472, 1473, 329}, 1: {2, 198, 1543, 1799, 1800, 585, 269, 909, 1361, 614, 1452, 1004, 109, 48, 1330, 1017, 317, 1855}, 2: {1, 198, 1543, 1799, 1800, 585, 269, 909, 1361, 614, 1452, 1004, 109, 48, 1330, 1017, 317, 1855}, 3: {1920, 382, 782, 1169, 1276, 663, 1180, 1566, 1952, 33, 1956, 1703, 1196, 1203, 1719, 185, 186, 1212, 1214, 1348, 454, 331, 1240, 1375, 1376, 100, 1509, 494, 241, 755, 1907, 380, 892, 894}, 4: {1088, 1795, 773, 137, 1225, 11, 202, 1612, 599, 600, 1627, 1889, 551, 1895, 425, 433, 1651, 629, 1782, 1783, 184, 1529, 1663}, 5: {66, 898, 804, 1892, 1128, 1545, 654, 1200, 466, 339, 1138, 1906, 1911, 830}, 6: {1409, 1673, 147, 403, 277, 537, 1564, 158, 1694, 1316, 165, 1444, 169, 43, 174, 1329, 1077, 695, 1594, 1595, 444, 1851, 1095, 73, 844, 1104, 1107, 86, 601, 1505, 1126, 1383, 744, 1645, 1009, 1269, 503}, 7: {1034, 1037, 1558, 1816, 1277, 31, 1058, 1828, 812, 1584, 568, 569, 1339, 574, 1096, 1608, 1098, 1610, 1102, 79, 1873, 597, 605, 862, 95, 1378, 356, 1639, 872, 

Let us now create a csv of edges connecting the ids that are coauthors.

In [33]:
coauthors_df = pd.DataFrame()
sources = []
targets = []
for key, val in source_to_related_sources.items():
    for item in val:
        sources.append(key)
        targets.append(item)

edge_name = ["coauthors"] * len(sources)

coauthors_df['source'] = sources
coauthors_df['relation'] = edge_name
coauthors_df['target'] = targets


coauthors_df.to_csv(path_to_folder + "authors.coauthors.authors.edges.csv", 
                    index=False)




In [34]:
authors = pd.read_csv(path_to_folder + "authors.nodes.csv")

In [35]:
authors

,id,label,PMIDS,affiliation
0,0,A M Maga,36802342;39554050,(Center for Developmental Biology and Regenera...
1,1,Aaron Goldman,38926365,"(Molecular and Cellular Oncogenesis Program, T..."
2,2,Aaron Havas,38926365,(Sanford Burnham Prebys Medical Discovery Inst...
3,3,Aaron W Puri,34862502,"(Department of Chemistry, University of Utah, ..."
4,4,Abhijith Asok,36350676,"(Microsoft Inc. Redmond, WA, USA.)"
...,...,...,...,...
1964,1964,Yu Jiang,30643251;36477530,"(Department of Public Health Sciences, College..."
1965,1965,Yufei Huang,38313267;38370127,"(Department of Medicine, University of Pittsbu..."
1966,1966,Yuin-Han Loh,37657444;39149248;39562549,"(Cell Fate Engineering and Therapeutics Lab, C..."
1967,1967,Brian E Cade,31964835;35927319;36303018;36477530;36564505,"(Division of Sleep and Circadian Disorders, Br..."


In [40]:
# Function to transform the name format
def transform_name(name):
    parts = name.split()
    last_name = parts[-1]  # Last part is the surname
    first_names = ' '.join(parts[:-1])  # Everything else is the first/middle names
    return f"{last_name}, {first_names}"

In [41]:
authors['Name'] = authors['label'].apply(transform_name)

In [42]:
authors

,id,label,PMIDS,affiliation,Name
0,0,A M Maga,36802342;39554050,(Center for Developmental Biology and Regenera...,"Maga, A M"
1,1,Aaron Goldman,38926365,"(Molecular and Cellular Oncogenesis Program, T...","Goldman, Aaron"
2,2,Aaron Havas,38926365,(Sanford Burnham Prebys Medical Discovery Inst...,"Havas, Aaron"
3,3,Aaron W Puri,34862502,"(Department of Chemistry, University of Utah, ...","Puri, Aaron W"
4,4,Abhijith Asok,36350676,"(Microsoft Inc. Redmond, WA, USA.)","Asok, Abhijith"
...,...,...,...,...,...
1964,1964,Yu Jiang,30643251;36477530,"(Department of Public Health Sciences, College...","Jiang, Yu"
1965,1965,Yufei Huang,38313267;38370127,"(Department of Medicine, University of Pittsbu...","Huang, Yufei"
1966,1966,Yuin-Han Loh,37657444;39149248;39562549,"(Cell Fate Engineering and Therapeutics Lab, C...","Loh, Yuin-Han"
1967,1967,Brian E Cade,31964835;35927319;36303018;36477530;36564505,"(Division of Sleep and Circadian Disorders, Br...","Cade, Brian E"


In [43]:
authors.to_csv(path_to_folder + "authors.nodes.csv")